<a href="https://colab.research.google.com/github/MaynineHuang/COMPSCI764_RAG/blob/main/notebooks/Retrieval_V1_WithExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/MaynineHuang/COMPSCI764_RAG.git
%cd COMPSCI764_RAG

Cloning into 'COMPSCI764_RAG'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 62 (delta 26), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 26.56 KiB | 2.66 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/COMPSCI764_RAG


In [2]:
!pip install -q datasets sentence-transformers

In [4]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

In [5]:
#load the data
hotpot = load_dataset(
        "hotpotqa/hotpot_qa",
        "distractor",
        split="validation[:5]"
         )
print(hotpot)

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 5
})


In [6]:
#load the embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
retrieval_model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2


In [7]:
example = hotpot[0]
print("Question:")
print(example["question"])
print("\nGold answer:")
print(example["answer"])
print("\nGold supporting facts:")
print(example["supporting_facts"])

Question:
Were Scott Derrickson and Ed Wood of the same nationality?

Gold answer:
yes

Gold supporting facts:
{'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}


In [8]:
#从这一步, get the question
titles = example["context"]["title"]
sentence_lists = example["context"]["sentences"]
passages = []
for title, sentences in zip(titles, sentence_lists):
     passage = title + ". " + " ".join(sentences)
     passages.append(passage)
print("Number of passages:", len(passages))

Number of passages: 10


In [9]:
#get the passages
question_embedding = retrieval_model.encode(
    example["question"],
    convert_to_tensor=True
)
passage_embeddings = retrieval_model.encode(
    passages,
    convert_to_tensor=True
)
print("Question embedding shape:", question_embedding.shape)
print("Passage embeddings shape:", passage_embeddings.shape)

Question embedding shape: torch.Size([384])
Passage embeddings shape: torch.Size([10, 384])


In [10]:
#Calculate cosine similarity
scores = util.cos_sim(
    question_embedding,
    passage_embeddings
)[0]
print(scores)

tensor([0.4753, 0.2636, 0.2922, 0.1315, 0.5520, 0.0636, 0.3287, 0.0818, 0.3027,
        0.0329])


In [11]:
#rank all passages
ranked_indices = scores.argsort(descending=True)
for rank, idx in enumerate(ranked_indices):
   idx = int(idx)
   print("\nRank:", rank + 1)
   print("Score:", round(float(scores[idx]), 4))
   print("Title:", titles[idx])


Rank: 1
Score: 0.552
Title: Ed Wood

Rank: 2
Score: 0.4753
Title: Ed Wood (film)

Rank: 3
Score: 0.3287
Title: Adam Collis

Rank: 4
Score: 0.3027
Title: Conrad Brooks

Rank: 5
Score: 0.2922
Title: Woodson, Arkansas

Rank: 6
Score: 0.2636
Title: Scott Derrickson

Rank: 7
Score: 0.1315
Title: Tyler Bates

Rank: 8
Score: 0.0818
Title: Sinister (film)

Rank: 9
Score: 0.0636
Title: Deliver Us from Evil (2014 film)

Rank: 10
Score: 0.0329
Title: Doctor Strange (2016 film)


In [12]:
#keep only the top 5 passages
TOP_K = 5
top_indices = ranked_indices[:TOP_K]
retrieved_titles = []
print("TOP 5 RETRIEVED PASSAGES")
for rank, idx in enumerate(top_indices):
  idx = int(idx)

  retrieved_titles.append(titles[idx])

  print("\nRank:", rank + 1)
  print("Score:", round(float(scores[idx]), 4))
  print("Title:", titles[idx])

TOP 5 RETRIEVED PASSAGES

Rank: 1
Score: 0.552
Title: Ed Wood

Rank: 2
Score: 0.4753
Title: Ed Wood (film)

Rank: 3
Score: 0.3287
Title: Adam Collis

Rank: 4
Score: 0.3027
Title: Conrad Brooks

Rank: 5
Score: 0.2922
Title: Woodson, Arkansas


In [13]:
#measure supporting-passage evidene recall
gold_support_titles = set(example["supporting_facts"]["title"])
retrieved_title_set = set(retrieved_titles)

found_titles = gold_support_titles.intersection(retrieved_title_set)

evidence_recall = len(found_titles) / len(gold_support_titles)

print("Gold supporting titles:", gold_support_titles)
print("Retrieved titles:", retrieved_title_set)
print("Found supporting titles:", found_titles)
print("Evidence recall:", evidence_recall)

Gold supporting titles: {'Ed Wood', 'Scott Derrickson'}
Retrieved titles: {'Adam Collis', 'Ed Wood', 'Ed Wood (film)', 'Conrad Brooks', 'Woodson, Arkansas'}
Found supporting titles: {'Ed Wood'}
Evidence recall: 0.5


In [15]:
#define the retrieve_top_k
def retrieve_top_k(example, model, top_k=5):
  question = example["question"]
  titles = example["context"]["title"]
  sentence_lists = example["context"]["sentences"]

  passages = [
      title + ". " + " ".join(sentences)
      for title, sentences in zip(titles, sentence_lists)
  ]
  question_embedding = model.encode(
      question,
      convert_to_tensor=True
  )

  passage_embeddings = model.encode(
      passages,
      convert_to_tensor=True
  )

  scores = util.cos_sim(
      question_embedding,
      passage_embeddings    )[0]

  ranked_indices = scores.argsort(descending=True)
  top_indices = ranked_indices[:top_k]

  results = []

  for rank, idx in enumerate(top_indices):
    idx = int(idx)
    results.append({
        "rank": rank + 1,
        "title": titles[idx],
        "passage": passages[idx],
        "score": float(scores[idx])
        })

  return results

In [16]:
#test the function on example1
results = retrieve_top_k(
    hotpot[0],
    retrieval_model,
    top_k=5
)
for result in results:
  print(
      result["rank"],
      round(result["score"], 4),
      result["title"]
  )

1 0.552 Ed Wood
2 0.4753 Ed Wood (film)
3 0.3287 Adam Collis
4 0.3027 Conrad Brooks
5 0.2922 Woodson, Arkansas


In [17]:
for i in range(5):
  example = hotpot[i]

  results = retrieve_top_k(
      example,
      retrieval_model,
      top_k=5
  )

  retrieved_titles = {
      result["title"]
      for result in results
  }

  gold_titles = set(
      example["supporting_facts"]["title"]
  )

  found = gold_titles.intersection(retrieved_titles)

  recall = len(found) / len(gold_titles)

  print("\n" + "=" * 70)
  print("EXAMPLE", i + 1)
  print("Question:", example["question"])
  print("Gold answer:", example["answer"])
  print("Gold supporting titles:", gold_titles)

  print("\nTop 5:")
  for result in results:
    print(
        result["rank"],
        "|",
        round(result["score"], 4),
        "|",
        result["title"]
    )

  print("\nEvidence recall:", recall)


EXAMPLE 1
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Gold supporting titles: {'Ed Wood', 'Scott Derrickson'}

Top 5:
1 | 0.552 | Ed Wood
2 | 0.4753 | Ed Wood (film)
3 | 0.3287 | Adam Collis
4 | 0.3027 | Conrad Brooks
5 | 0.2922 | Woodson, Arkansas

Evidence recall: 0.5

EXAMPLE 2
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold answer: Chief of Protocol
Gold supporting titles: {'Shirley Temple', 'Kiss and Tell (1945 film)'}

Top 5:
1 | 0.6683 | Kiss and Tell (1945 film)
2 | 0.5741 | A Kiss for Corliss
3 | 0.5452 | Meet Corliss Archer (TV series)
4 | 0.3984 | Janet Waldo
5 | 0.3342 | Meet Corliss Archer

Evidence recall: 0.5

EXAMPLE 3
Question: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Gold answer: Animorphs
Gold supporting titles: {'The Hork-Bajir Chronicles', 'Anim